## **Climate Change Dataset**
created by: [aahan-kumar](https://github.com/aahan-kumar) @ github  
‎  
This project uses real world temperature anomaly data.  

Temperature Anomaly is an important metric used by scientists and environmental organisations to track Global Warming and Climate Change.  

It is calculated as the difference between an observed temperature and a long term avg. (baseline) temperature.

The data contains different places and their temperature anomalies from 1940 onwards.  
‎  
___  

(Source: OWID 2026, https://ourworldindata.org/grapher/annual-temperature-anomalies.csv)  


In [ ]:
#install libraries in notebook
!pip -q install pandas numpy matplotlib scikit-learn

---
1)
This step imports all required libraries & module(s)  
It gets dataset from source and uses pandas to read its csv contents.  
It uses pandas to create DataFrame based on csv info.
It also provides a brief overview of dataset, e.g. no. of rows, columns, etc.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

#owid dataset for temp anomaly data of various places
df = pd.read_csv(
    "https://ourworldindata.org/grapher/annual-temperature-anomalies.csv?v=1&csvType=full&useColumnShortNames=false",
    storage_options={'User-Agent': 'Our World In Data data fetch/1.0'}
)

df = df[["Entity", "Year", "Temperature anomaly"]].dropna()

#renaming default column in DataFrame
df = df.rename(columns={"Entity": "Place", "Temperature anomaly": "temperature_anomaly"})

#keeping 2025 as the final analysis year for this project
finalyear = 2025

print('Success! Data loaded!')
print("Welcome to my project...")
print("- This database contains yearwise temperature anomaly data from 1940 onwards")
print('- Temp. anomaly is very important concept in climate change','\n- It is calculated as difference b/w observed temp. - baseline temp (for a place)')
print("- It includes places, their years, and temperature anomalies.")
print("\n- Some details about this dataset:- \n * Number of records:", len(df))
print(" * Year range:", df["Year"].min(), "to", df["Year"].max())
print(" * Analysis year:", finalyear)
print(" * Columns:", list(df.columns))

---
2)
We plot line chart of Temp. Anomaly (in deg C) against the years 1940 onwards using matplotlib.

Here we use the **World** entry already present in the dataset instead of taking a simple average of all places.  
This gives the global anomaly series supplied by the dataset.

---


In [ ]:
world = df[(df["Place"] == "World") & (df.Year <= finalyear)].sort_values("Year")

plt.figure(figsize=(11, 5))
plt.plot(world.Year, world.temperature_anomaly)
plt.title("Global Climate Change Trend")
plt.xlabel("Year")
plt.ylabel("Temp. anomaly (°C)")
plt.grid(alpha=0.25) #opacity of grid
plt.tight_layout()
plt.show()

---
3)
We draw bar graph of places w/ highest anomalies in 2025 using matplotlib.  

This shows the places with the highest temperature anomaly in that year. It is a **one-year anomaly ranking**, not a ranking of fastest warming places.  

It also provides info about India's position in the anomaly ranks of all listed places/entities.

---


In [ ]:
ranked2025 = df[df.Year == finalyear].sort_values(
    "temperature_anomaly", ascending=False
).reset_index(drop=True)
hot2025 = ranked2025.head(10)

plt.figure(figsize=(10, 6))
plt.barh(hot2025.Place[::-1], hot2025.temperature_anomaly[::-1])
plt.xlabel("Temperature anomaly (°C)")
plt.title(f"Top 10 Climate Hotspot in {finalyear}")
plt.tight_layout()
plt.show()

print("Top 10 places with highest anomaly:")
print(hot2025[["Place", "temperature_anomaly"]].to_string(index=False))

india_rank = ranked2025.index[ranked2025.Place == "India"]
if len(india_rank) > 0:
    print("\nIndia's anomaly rank in year 2025:", india_rank[0] + 1)
else:
    print("\nIndia was not found in the 2025 data.")

---
4)
We use numpy and sklearn to perform linear fitting and trend analysis on our dataframe.  
We use sklearn KMeans module to do KMeans clustering on our data for warming patterns. We use k=4 for low, moderate, higher, and very high warming patterns.  

The clustering uses the same last decade (2016-2025) for recent anomaly, variation, and trend.  The three features are standardised before KMeans so that one feature does not dominate the distance calculation just because of its numerical scale.  

This also tells India's climate pattern in the last decade. The names below are simple labels for the clusters, not official climate categories.

---


In [ ]:
recent = df[(df.Year >= finalyear - 9) & (df.Year <= finalyear)].copy()

recentyears = recent.groupby("Place")["temperature_anomaly"].agg(["mean", "std"]).rename(columns={"mean": "recent", "std": "variation"})
trend = recent.groupby("Place").apply(
    lambda x: np.polyfit(x.Year, x.temperature_anomaly, 1)[0],
    include_groups=False
).rename("trend")
features = recentyears.join(trend).dropna()

scaler = StandardScaler()
scaled = scaler.fit_transform(features[["recent", "variation", "trend"]])

kmeans = KMeans(n_clusters=4, random_state=1, n_init=10)
features["cluster"] = kmeans.fit_predict(scaled)

centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=["recent", "variation", "trend"]
)
order = centers.recent.sort_values().index
names = {
    order[0]: "Lower anomaly pattern",
    order[1]: "Moderate anomaly pattern",
    order[2]: "Higher anomaly pattern",
    order[3]: "Very High anomaly pattern"
}
features["pattern"] = features.cluster.map(names)

print("Using KMeans Clustering Algorithm w/ K=4 we get these climate patterns:")
print(features.pattern.value_counts())
print("\nFirst few clustered places:")
print(features.head(6).to_string())

print("\nIndia's climate pattern in last decade (2016-25):")
print(features.loc[["India"]])

---
5)
We use matplotlib to create a scatter plot for the clustering visualisation.  
Appropriate legend and keys are also shown.

---


In [ ]:
plt.figure(figsize=(10, 6))
for cluster in range(4):
    data = features[features["cluster"] == cluster]
    plt.scatter(
        data["recent"],
        data["trend"],
        label=names[cluster]
    )
plt.xlabel("Recent years temperature anomaly (2016-25)")
plt.ylabel("Warming trend (°C / year)")
plt.title("KMeans Clusters")
plt.legend()
plt.grid(alpha=0.25) #opacity of grid
plt.tight_layout()
plt.show()

---
6)
We now plot a line chart of temperature anomalies in degree Celsius against the years for India.  
We also plot a 5 year rolling average in this duration.  

Then using linear regression, we create a simple trend extrapolation for temperature anomalies from years 2026 to 2050.  

We also calculate MAE (mean absolute error) and R^2 score.

This is a mathematical extrapolation from the data and should not be treated as a full climate forecast. (R^2 is -ve)

---


In [ ]:
country = "India"
d = df[(df.Place == country) & (df.Year <= finalyear)].sort_values("Year").copy()
x = d[["Year"]].values
y = d["temperature_anomaly"].values
train = int(len(d) * 0.8)

model = LinearRegression()
model.fit(x[:train], y[:train])
test = model.predict(x[train:])
mae = mean_absolute_error(y[train:], test)
r2 = r2_score(y[train:], test)

#fit on all available India data before making the 2026-2050 extrapolation
model.fit(x, y)
future = np.arange(finalyear + 1, 2051).reshape(-1, 1)
fourcast = model.predict(future)

trend10 = model.coef_[0] * 10
pattern = features.loc[country, "pattern"]
recent_trend10 = features.loc[country, "trend"] * 10

print("Country:", country)
print("Climate trend pattern:", pattern)
print(f"Recent 10 year trend: {recent_trend10:.2f} °C / decade")
print(f"Long-term linear trend: {trend10:.2f} °C / decade")
print(f"Model MAE: {mae:.2f} °C")
print(f"Model R^2: {r2:.2f}")

if r2 < 0:
    print("Note: R^2 is below 0, so this linear model is not a strong fit for the held-out years.")

print("\nTrend extrapolation till 2050:")
print('-------------------')
for year, value in zip(future.ravel(), fourcast):
    print(f"{year}: {value:.2f} °C")
print('-------------------')

plt.figure(figsize=(11, 5))
plt.plot(d.Year, d.temperature_anomaly, label="Actual trend")
plt.plot(d.Year, d.temperature_anomaly.rolling(5).mean(), label="5 year rolling avg")
plt.plot(future.ravel(), fourcast, "o--", label="Linear trend extrapolation")
plt.title(f"{country}: Climate Trend and Linear Regression")
plt.xlabel("Year")
plt.ylabel("Temperature anomaly (°C)")
plt.legend()
plt.grid(alpha=0.25) #opacity of grid
plt.tight_layout()
plt.show()

---
7)
Now, we extract a list of the top climate hotspots and filter our dataset to analyze data from 2011 to 2025.  
Now we transform this filtered, multi-year data into a spreadsheet-style pivot grid mapping locations against years.  
Finally, it uses matplotlib to generate a structured climate heatmap with a dedicated color bar scale and rotated axes, visually tracking how temperature anomalies have shifted across these regions over the last 15 years.  

---


In [ ]:
head = hot2025.Place.tolist()
heat = df[(df.Place.isin(head)) & (df.Year >= 2011) & (df.Year <= finalyear)].pivot(
    index="Place", columns="Year", values="temperature_anomaly"
)
plt.figure(figsize=(12, 6))
plt.imshow(heat, aspect="auto")
plt.colorbar(label="Temperature anomaly (°C)")
plt.yticks(range(len(heat.index)), heat.index)
plt.xticks(range(len(heat.columns)), heat.columns, rotation=45)
plt.title("15 year Climate Hotspot Heatmap")
plt.tight_layout()
plt.show()

---
8)
Some final points from this analysis/ Conclusion:-  
- The global trend is taken from the World entry in the dataset rather than averaging all listed places.  
- The 2025 hotspot graph shows highest anomaly in one year; it does not by itself show the fastest warming places.  
- KMeans groups places using recent anomaly, variation and recent linear trend. These labels are made for this project and are not official climate classifications.  
- The India regression is useful for seeing the long-term linear trend, but the 2026-2050 values are only trend extrapolation.  
- A more complete climate model would need many other factors and is outside the scope of this project.

---
